# MLP Dengue predict

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

ModuleNotFoundError: No module named 'google'

In [ ]:
drive.mount('/content/drive')

Load dataset

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/DENGUE/databases/dataset.csv', low_memory=True)
data.head()

### To separate data from target

In [ ]:
x = data.drop(columns=['Gender', 'Area', 'AreaType', 'HouseType',
                       'District', 'Outcome'])
y = data['Outcome']

In [ ]:
for col in x.columns:
  print(x.value_counts(col))
  print("")
print(y.value_counts())

### To separate train from test

### Normalize atributes

In [ ]:
scaler = StandardScaler()
x_columns_to_normalize = x.drop(columns=['Headache', 'Retro_Orbital_Pain', 'NS1',
                                      'IgG', 'IgM', 'Myalgia', 'Rash', 'Joint_Pain'])

original_column_names = x_columns_to_normalize.columns

x_columns_normalize = scaler.fit_transform(x_columns_to_normalize)
x_columns_normalize = pd.DataFrame(x_columns_normalize, columns=original_column_names)
x_columns_normalize.head()

In [ ]:
non_normalized_cols = ['Headache', 'Retro_Orbital_Pain', 'NS1', 'IgG', 'IgM', 'Myalgia', 'Rash', 'Joint_Pain']

x_categorical = x[non_normalized_cols]

x = pd.concat([x_categorical, x_columns_normalize], axis=1)
x.head()

In [ ]:
x.info()

### Create columns severe, moderate and no_severe_moderate

In [ ]:
x['Severe'] = x['Joint_Pain'].apply(lambda x: 1 if x == 'Severe' else 0)
x['Moderate'] = x['Joint_Pain'].apply(lambda x: 1 if x == 'Moderate' else 0)
x['No_Severe_Moderate'] = x['Joint_Pain'].apply(lambda x: 0 if x != 'Severe' and x != 'Moderate'  else 1)
x.head()

In [ ]:
y = x['No_Severe_Moderate']
x = x.drop(columns=['Joint_Pain', 'NS1', 'IgG', 'IgM', 'Severe', 'Moderate', 'No_Severe_Moderate', 'Age'])
x.head()

In [ ]:
x.columns

In [ ]:
y.head()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
import torch.nn as nn

class MLPregression(nn.Module):
    def __init__(self, input_size):
        super(MLPregression, self).__init__()
        self.input = nn.Linear(input_size, 64)
        self.hidden = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.input(x))
        x = self.relu(self.hidden(x))

        x = self.sigmoid(self.output(x))
        return x


In [ ]:
x_train = torch.tensor(x_train.values, dtype = torch.float32)
x_test = torch.tensor(x_test.values, dtype = torch.float32)
y_train = torch.tensor(y_train.values, dtype = torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test.values, dtype = torch.float32).unsqueeze(1)

In [ ]:
model = MLPregression(input_size=x_train.shape[1])
criterion = nn.BCELoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=0.001)

epocs = 2000
train_loss = np.zeros(epocs)

for epoch in range(epocs):
    model.train()
    optimizer.zero_grad()

    outputs = model(x_train)
    pesos = torch.where(y_train == 1, 15.0, 1.0)

    element_wise_loss = criterion(outputs, y_train)

    weighted_loss = element_wise_loss * pesos
    loss = torch.mean(weighted_loss)

    loss.backward()

    optimizer.step()
    train_loss[epoch] = loss.item()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()

with torch.no_grad():
    test_outputs = model(x_test)

    previsoes = (test_outputs > 0.5).float()

y_true = y_test.numpy()
y_pred = previsoes.numpy()


acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("=== RESULTADOS DA AVALIAÇÃO NO CONJUNTO DE TESTE ===")
print(f"Acurácia (Acertos totais) : {acc:.4f} ({acc*100:.2f}%)")
print(f"Precisão (Qualidade dos alarmes) : {prec:.4f}")
print(f"Recall   (Capacidade de achar os positivos) : {rec:.4f}")
print(f"F1-Score (Equilíbrio Precisão/Recall) : {f1:.4f}\n")


cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'},
            xticklabels=['Negativo (0)', 'Positivo (1)'],
            yticklabels=['Negativo (0)', 'Positivo (1)'])

plt.title('Matriz de Confusão - Modelo Dengue', pad=15, fontsize=14, fontweight='bold')
plt.xlabel('Previsão do Modelo', fontsize=12, fontweight='bold', labelpad=10)
plt.ylabel('Realidade (Gabarito)', fontsize=12, fontweight='bold', labelpad=10)
plt.tight_layout()
plt.show()